# Testing

In [1]:
# Imports

import time

import numpy as np
import roboticstoolbox as rtb
import spatialgeometry as sg
import spatialmath as sm
import swift

Formula: $\mathbf{p}^W = \mathbf{T}^W_T,\mathbf{R}_z(\varphi),\mathbf{p}^B$

In [8]:
# Initializations
T_table = sm.SE3(0.75, 0.0, 0.2)   # This should be the center of the table-top (where the marble block sits)
BLOCK_SIDE = 0.3                  # [m] width and depth of the block (for now, could change)
BLOCK_HEIGHT = 0.45               # [m]
MARBLE = [0.92, 0.92, 0.88, 1.00]

In [ ]:
# Block
def make_blocks(size: float) -> list[np.ndarray]:
    """Centers of the cuboid in the block frame (origin at the center of the bottom face)"""
    n = round(BLOCK_SIDE / size) 
    m = round(BLOCK_HEIGHT / size)
    centers = []
    
    for i in range(n):
        for j in range(n):
            for k in range(m):
                centers.append(np.array([(i - (n - 1) / 2) * size, 
                                         (j - (n - 1) / 2) * size, 
                                         (k - (n - 0.5) / 2) * size]))
    return centers

# Table (where the block goes)
def place(cubes: list[sg.Cuboid], centers: list[np.ndarray], phi: float) -> None:
    """Same block pose for all cubes, then each one at its own place inside the block"""
    T_block = T_table * sm.SE3.Rz(phi)
    for cube, p in zip(cubes, centers):
        cube.T = T_block * sm.SE3(p)

In [9]:
# Test
def test_scene(size: float, n_steps: int = 100, dt: float = 0.05) -> dict[str, float]:
    env = swift.Swift()
    env.launch(realtime=False, browser=None)    # Not realtime in this case because we are measuring a single step's speed.
    
    panda = rtb.models.Panda()
    panda.q = panda.qr
    env.add(panda)
    env.add(sg.Cylinder(radius=0.22, length=0.2, pose=sm.SE3(0.75, 0, 0.1), color=[0.4, 0.3, 0.2, 1.0]))
    
    # Chisel held by the "gripper", along the z axis of the hand
    chisel = sg.Cylinder(radius=0.008, length=0.15, color=[0.3, 0.3, 0.3, 1.0])
    chisel.T = panda.fkine(panda.q) * sm.SE3(0, 0, 0.03)
    env.add(chisel)
    
    centers = make_blocks(size)
    cubes = [sg.Cuboid(scale=[size] * 3, color=MARBLE) for _ in centers]
    place(cubes, centers, 0.0)
    
    t0 = time.perf_counter()
    ids = [env.add(cube) for cube in cubes]
    t_add = time.perf_counter() - t0
    
    # TODO: Continue from here
    def mean_step_time(rotate: bool, remove: bool) -> float:
        times = []
        phi = 0.0
        for _ in range(n_steps):
            if rotate:
                phi += 0.02
                place(cubes, centers, phi)
            if remove and cubes:
                env.remove(ids.pop())
                cubes.pop()
                centers.pop()
            t0 = time.perf_counter()
            env.step(dt)
            times.append(time.perf_counter() - t0)
        return float(np.mean(times))
    
    result = {
        "cubes": len(cubes),
        "add [s]": t_add,
        "step, still [s]": mean_step_time(rotate=False, remove=False),
        "step, rotating [s]": mean_step_time(rotate=True, remove=False),
        "step, removing [s]": mean_step_time(rotate=False, remove=True),
    }
    env.close()
    return result

In [10]:
# Test sizes from big to small ones...
for size in [0.03, 0.02]:
    print(f"size {size} m:", test_scene(size))

Task was destroyed but it is pending!
task: <Task pending name='Task-17483' coro=<serve.<locals>.protocol_handler() running at c:\Users\aland\.venvs\rca\Lib\site-packages\websockets\asyncio\server.py:747> wait_for=<Future pending cb=[Task.task_wakeup()]>>
Task was destroyed but it is pending!
task: <Task pending name='Task-17484' coro=<Connection.keepalive() running at c:\Users\aland\.venvs\rca\Lib\site-packages\websockets\asyncio\connection.py:815> wait_for=<Future pending cb=[Task.task_wakeup()]>>
Task was destroyed but it is pending!
task: <Task pending name='Task-59329' coro=<SwiftSocket.producer() running at c:\Users\aland\.venvs\rca\Lib\site-packages\swift\SwiftRoute.py:355> wait_for=<Future finished result=[False, ['sim_time', 15.000000000000078]]>>
Task was destroyed but it is pending!
task: <Task pending name='Task-59330' coro=<Connection.wait_closed() running at c:\Users\aland\.venvs\rca\Lib\site-packages\websockets\asyncio\connection.py:657> wait_for=<Future pending cb=[shie

size 0.03 m: {'cubes': 1500, 'add [s]': 5.175617000000784, 'step, still [s]': 0.009373705999969388, 'step, rotating [s]': 0.020554938999557636, 'step, removing [s]': 0.020309806999648572}


Task was destroyed but it is pending!
task: <Task pending name='Task-59338' coro=<serve.<locals>.protocol_handler() running at c:\Users\aland\.venvs\rca\Lib\site-packages\websockets\asyncio\server.py:747> wait_for=<Future pending cb=[Task.task_wakeup()]>>
Task was destroyed but it is pending!
task: <Task pending name='Task-59339' coro=<Connection.keepalive() running at c:\Users\aland\.venvs\rca\Lib\site-packages\websockets\asyncio\connection.py:815> wait_for=<Future pending cb=[Task.task_wakeup()]>>
Task was destroyed but it is pending!
task: <Task pending name='Task-73208' coro=<SwiftSocket.producer() running at c:\Users\aland\.venvs\rca\Lib\site-packages\swift\SwiftRoute.py:355> wait_for=<Future pending cb=[_chain_future.<locals>._call_check_cancel() at C:\Users\aland\AppData\Roaming\uv\python\cpython-3.12-windows-x86_64-none\Lib\asyncio\futures.py:389, Task.task_wakeup()]>>
Task was destroyed but it is pending!
task: <Task pending name='Task-73209' coro=<Connection.wait_closed() run

size 0.02 m: {'cubes': 4950, 'add [s]': 22.685206500005734, 'step, still [s]': 0.0985508999996091, 'step, rotating [s]': 0.06927158199971017, 'step, removing [s]': 0.058835996000198065}


## Conclusions

- Swift updates every object at every step, so what matters is the number of cubes. The medium size runs smoothly, the small one is the limit: we use the medium size.
- Remove cubes by the id that `env.add` returns. Removing by object searches the whole list every time.
- Turn the table only between sectors, not while the arm is cutting.
- Don't load the inside of the statue, it is never seen. The block is the full box around the statue, minus the statue's interior.
- Keep the block far enough from the robot, so the hand at `qr` is never inside it, even when the table turns.
- The chisel must be thinner than a cube and have a visible tip: that tip is the point we control.